# 8 Ablation Experiments — based on Dean's exp 16

Each cell is **self-contained** — paste it into a SEPARATE Colab Pro tab.

## How to use

1. Open up to 4 Colab Pro tabs at once (T4 GPU each)
2. In each tab: copy/paste the SETUP cell, then ONE experiment cell
3. Click Run — leave tab open until done (~3-4 hr)
4. Each experiment saves `<EXP_NAME>_best.h5` to Drive `picar_models/`

## Tonight (batch 1, kick off 1-4)
- Cell A1: `45_baseline16_clean`
- Cell A2: `46_huber_bce`
- Cell A3: `47_crop120_30`
- Cell A4: `48_progressive`

## Tomorrow morning (batch 2, kick off 5-8)
- Cell A5: `49_cutout_smaller`
- Cell A6: `50_cutout_aggressive`
- Cell A7: `51_dense_smaller`
- Cell A8: `52_bn_locked`

## SETUP cell (paste into EVERY new tab first)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
ZIP_PATH    = '/content/drive/MyDrive/picar_data.zip'
SCRIPT_PATH = '/content/drive/MyDrive/train_ablation.py'

assert os.path.exists(ZIP_PATH),    f'Upload picar_data.zip to Drive root: {ZIP_PATH}'
assert os.path.exists(SCRIPT_PATH), f'Upload train_ablation.py to Drive root: {SCRIPT_PATH}'

# Unzip data (skip if already done in this runtime)
if not os.path.exists('/content/PiCar/data/training_data/training_data'):
    !mkdir -p /content/PiCar
    !cd /content/PiCar && unzip -q -o /content/drive/MyDrive/picar_data.zip
    print('✅ Unzipped data')
else:
    print('⏭️  Data already unzipped')

# Copy training script
import shutil
os.makedirs('/content/PiCar/src', exist_ok=True)
shutil.copy(SCRIPT_PATH, '/content/PiCar/src/train_ablation.py')
print('✅ Script copied')

# Verify GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
assert gpus, 'No GPU! Runtime → Change runtime type → T4 GPU'
print(f'✅ GPU ready: {gpus}')

# Make sure data/corrections.csv exists in unzipped data
import os
if not os.path.exists('/content/PiCar/data/corrections.csv'):
    # If not in zip, copy from Drive (you should upload it)
    if os.path.exists('/content/drive/MyDrive/corrections.csv'):
        os.makedirs('/content/PiCar/data', exist_ok=True)
        shutil.copy('/content/drive/MyDrive/corrections.csv', '/content/PiCar/data/corrections.csv')
        print('✅ Corrections.csv copied from Drive')
    else:
        print('⚠️  corrections.csv missing — corrections will be skipped')

# Make sure data/bad_images.csv is the LATEST (366 entries)
if os.path.exists('/content/drive/MyDrive/bad_images.csv'):
    shutil.copy('/content/drive/MyDrive/bad_images.csv', '/content/PiCar/data/bad_images.csv')
    print('✅ bad_images.csv updated from Drive')

# wandb login
!pip install wandb -q
import wandb
wandb.login()

# Drive save dir
DRIVE_SAVE_DIR = '/content/drive/MyDrive/picar_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'✅ Drive save dir ready: {DRIVE_SAVE_DIR}')

# Keep-alive cell (in background) — fights Colab idle timeout
import threading, time
from IPython.display import display, Javascript
def keep_alive():
    while True:
        time.sleep(600)
        try:
            display(Javascript('console.log("alive")'))
        except: pass
threading.Thread(target=keep_alive, daemon=True).start()
print('✅ Keep-alive started (every 10 min)')

## Cell A1 — `45_baseline16_clean` (BATCH 1)

Dean's exp 16 architecture + corrections.csv + updated bad_images + 200 epochs.

In [ ]:
import os, json
EXP_NAME = '45_baseline16_clean'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'Dean exp 16 architecture + corrections.csv + 366-bad-images + 200 epochs',
    'EPOCHS_FINETUNE': 195,  # 5 + 195 = 200 total
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

# Save best to Drive after training
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src):
    shutil.copy(src, dst)
    print(f'✅ Saved {dst}')

## Cell A2 — `46_huber_bce` (BATCH 1)

Same as baseline + Huber loss for angle + BCE for speed (binary).

In [ ]:
import os, json
EXP_NAME = '46_huber_bce'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + Huber angle + BCE speed (robust to label noise)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'huber',
    'SPEED_AS_CLASSIFICATION': True,    # → BCE for speed
    'USE_CLEAN_DATA': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A3 — `47_crop120_30` (BATCH 1)

Same as baseline + heavier top crop (120 instead of 110).

In [ ]:
import os, json
EXP_NAME = '47_crop120_30'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + crop top 120 (was 110)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'CROP_TOP_PIXELS': 120,             # ← only change
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A4 — `48_progressive` (BATCH 1)

Same as baseline + progressive unfreezing (1 block at a time, smoother optimization).

In [ ]:
import os, json
EXP_NAME = '48_progressive'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + progressive unfreezing (smoother optim)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'USE_PROGRESSIVE_UNFREEZING': True,    # NEW flag, see script
    'PROGRESSIVE_LR_DECAY': 0.85,
    'PROGRESSIVE_EPOCHS_PER_STEP': 5,      # 5 epochs per block unfrozen
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A5 — `49_cutout_smaller` (BATCH 2 — tomorrow morning)

Smaller cutout (10/30 was 10/50).

In [ ]:
import os, json
EXP_NAME = '49_cutout_smaller'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + cutout 10/30 (smaller, was 10/50)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'AUG_CUTOUT_MIN_PIX': 10, 'AUG_CUTOUT_MAX_PIX': 30,    # ← smaller
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A6 — `50_cutout_aggressive` (BATCH 2)

More aggressive cutout (15/60, prob 0.7).

In [ ]:
import os, json
EXP_NAME = '50_cutout_aggressive'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + cutout 15/60 prob 0.7 (more aggressive)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'AUG_CUTOUT_PROB': 0.7,
    'AUG_CUTOUT_MIN_PIX': 15, 'AUG_CUTOUT_MAX_PIX': 60,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A7 — `51_dense_smaller` (BATCH 2)

Smaller dense head (128→64 was 256→128).

In [ ]:
import os, json
EXP_NAME = '51_dense_smaller'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + dense head 128→64 (was 256→128)',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'DENSE_UNITS_1': 128, 'DENSE_UNITS_2': 64,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A8 — `52_bn_locked` (BATCH 2)

Same as baseline + explicit BatchNorm lock during fine-tune.

In [ ]:
import os, json
EXP_NAME = '52_bn_locked'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'baseline16 + explicit BatchNorm lock during finetune',
    'EPOCHS_FINETUNE': 195,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6, 'NUM_ATTN_BLOCKS': 2,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'LOCK_BATCHNORM': True,    # NEW flag
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log
import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')